<a href="https://colab.research.google.com/github/King1oo1/DEEPGUARD/blob/main/training_deepfake_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("muhammadbilal6305/200k-real-vs-ai-visuals-by-mbilal")

print("Path to dataset files:", path)

100%|██████████| 1.82G/1.82G [00:26<00:00, 73.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/muhammadbilal6305/200k-real-vs-ai-visuals-by-mbilal/versions/4


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ayushmandatta1/deepdetect-2025")

print("Path to dataset files:", path)

100%|██████████| 3.23G/3.23G [00:49<00:00, 69.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ayushmandatta1/deepdetect-2025/versions/1


In [3]:
# =====================================================================
# DeepGuard – Train on DeepDetect-2025 + GRAVEX-200K
# Handles both folder structures automatically
# =====================================================================

# 1. Install & Import
!pip install torch torchvision timm pillow tqdm kagglehub -q

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset, Dataset
import timm
from tqdm import tqdm
import os
import shutil
import pandas as pd
from PIL import Image
import kagglehub
from google.colab import files

# 2. Download datasets
print("📥 Downloading DeepDetect-2025...")
deepdetect_path = kagglehub.dataset_download("ayushmandatta1/deepdetect-2025")
print(f"DeepDetect path: {deepdetect_path}")

print("📥 Downloading GRAVEX-200K...")
gravex_path = kagglehub.dataset_download("muhammadbilal6305/200k-real-vs-ai-visuals-by-mbilal")
print(f"GRAVEX path: {gravex_path}")

# 3. Explore structures and for debugging
print("\n🔍 DeepDetect structure:")
!ls -R "{deepdetect_path}" | head -30
print("\n🔍 GRAVEX structure:")
!ls -R "{gravex_path}" | head -30

# 4. Create a unified dataset folder
base_dir = "/content/unified_dataset"
os.makedirs(os.path.join(base_dir, "train", "real"), exist_ok=True)
os.makedirs(os.path.join(base_dir, "train", "fake"), exist_ok=True)
os.makedirs(os.path.join(base_dir, "val", "real"), exist_ok=True)
os.makedirs(os.path.join(base_dir, "val", "fake"), exist_ok=True)

# =====================================================================
# 5. Process DeepDetect-2025 split into train/test
# =====================================================================
print("\n📂 Processing DeepDetect-2025...")
deep_train_real = os.path.join(deepdetect_path, "ddata", "train", "real")
deep_train_fake = os.path.join(deepdetect_path, "ddata", "train", "fake")
deep_test_real = os.path.join(deepdetect_path, "ddata", "test", "real")
deep_test_fake = os.path.join(deepdetect_path, "ddata", "test", "fake")

def copy_images(src_dir, dest_dir, max_images=None):
    if not os.path.exists(src_dir):
        print(f"⚠️ Source not found: {src_dir}")
        return 0
    images = [f for f in os.listdir(src_dir) if f.lower().endswith(('.jpg','.png','.jpeg'))]
    if max_images:
        images = images[:max_images]
    for img in images:
        shutil.copy(os.path.join(src_dir, img), os.path.join(dest_dir, img))
    return len(images)

# Copy DeepDetect training data
deep_train_real_count = copy_images(deep_train_real, os.path.join(base_dir, "train", "real"))
deep_train_fake_count = copy_images(deep_train_fake, os.path.join(base_dir, "train", "fake"))
# Copy DeepDetect test data as validation
deep_val_real_count = copy_images(deep_test_real, os.path.join(base_dir, "val", "real"))
deep_val_fake_count = copy_images(deep_test_fake, os.path.join(base_dir, "val", "fake"))

print(f"DeepDetect: train real={deep_train_real_count}, train fake={deep_train_fake_count}")
print(f"DeepDetect: val real={deep_val_real_count}, val fake={deep_val_fake_count}")

# =====================================================================
# 6. Process GRAVEX-200K (uses CSV label files)
# =====================================================================
print("\n📂 Processing GRAVEX-200K...")

# Find the actual image directories
# The structure shows my_real_vs_ai_dataset/my_real_vs_ai_dataset/ with 'real' and 'ai_images' folders
gravex_root = os.path.join(gravex_path, "my_real_vs_ai_dataset", "my_real_vs_ai_dataset")
real_dir = os.path.join(gravex_root, "real")
fake_dir = os.path.join(gravex_root, "ai_images")

# Load CSV files for splits
train_csv = os.path.join(gravex_root, "train_labels.csv")
val_csv = os.path.join(gravex_root, "val_labels.csv")

def copy_from_csv(csv_path, src_root, dest_dir, label_column, image_column="filename"):
    if not os.path.exists(csv_path):
        print(f"⚠️ CSV not found: {csv_path}")
        return 0
    df = pd.read_csv(csv_path)
    count = 0
    for idx, row in df.iterrows():
        img_name = row[image_column]
        label = row[label_column]
        # Determine if real or fake
        if label in ["real", "Real", 0]:
            target_class = "real"
        elif label in ["fake", "Fake", "ai", 1]:
            target_class = "fake"
        else:
            continue
        src_path = os.path.join(src_root, img_name)
        if os.path.exists(src_path):
            shutil.copy(src_path, os.path.join(dest_dir, target_class, img_name))
            count += 1
    return count

# If CSV files exist, use them; otherwise fall back to copying all images
if os.path.exists(train_csv) and os.path.exists(val_csv):
    print("Using CSV splits from GRAVEX")
    # Train
    gravex_train_real = copy_from_csv(train_csv, real_dir, base_dir, "real", "filename")
    gravex_train_fake = copy_from_csv(train_csv, fake_dir, base_dir, "fake", "filename")
    # Validation
    gravex_val_real = copy_from_csv(val_csv, real_dir, base_dir, "real", "filename")
    gravex_val_fake = copy_from_csv(val_csv, fake_dir, base_dir, "fake", "filename")
else:
    # Fallback: split 80/20 manually
    print("CSV splits not found – copying all images and splitting manually")
    all_real = [os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.endswith(('.jpg','.png'))]
    all_fake = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if f.endswith(('.jpg','.png'))]
    from sklearn.model_selection import train_test_split
    train_real, val_real = train_test_split(all_real, test_size=0.2, random_state=42)
    train_fake, val_fake = train_test_split(all_fake, test_size=0.2, random_state=42)
    for src in train_real: shutil.copy(src, os.path.join(base_dir, "train", "real"))
    for src in val_real: shutil.copy(src, os.path.join(base_dir, "val", "real"))
    for src in train_fake: shutil.copy(src, os.path.join(base_dir, "train", "fake"))
    for src in val_fake: shutil.copy(src, os.path.join(base_dir, "val", "fake"))
    gravex_train_real = len(train_real)
    gravex_train_fake = len(train_fake)
    gravex_val_real = len(val_real)
    gravex_val_fake = len(val_fake)

print(f"GRAVEX: train real={gravex_train_real}, train fake={gravex_train_fake}")
print(f"GRAVEX: val real={gravex_val_real}, val fake={gravex_val_fake}")

# =====================================================================
# 7. Create PyTorch Datasets
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🔥 Using device: {device}")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(base_dir, "train"), transform=train_transform)
val_dataset = datasets.ImageFolder(os.path.join(base_dir, "val"), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ Final training samples: {len(train_dataset)}")
print(f"✅ Final validation samples: {len(val_dataset)}")
print(f"Classes: {train_dataset.classes}")

# =====================================================================
# 8. Model (EfficientNet-B0 with class weights)
# =====================================================================
model = timm.create_model('efficientnet_b0', pretrained=True)
model.classifier = nn.Linear(model.classifier.in_features, 2)
model = model.to(device)

# Optional class weights (if imbalance)
class_counts = [len(train_dataset.targets) - sum(train_dataset.targets), sum(train_dataset.targets)]
class_weights = [max(class_counts)/c for c in class_counts]
class_weights = torch.tensor(class_weights).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# =====================================================================
# 9. Training Loop
# =====================================================================
epochs = 10
best_val_acc = 0.0

for epoch in range(epochs):
    model.train()
    correct = 0
    total = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()
        loop.set_postfix(loss=loss.item(), acc=correct/total)
    train_acc = correct / total
    print(f"Train accuracy: {train_acc:.4f}")

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        loop = tqdm(val_loader, desc="Validation")
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
            loop.set_postfix(acc=correct/total)
    val_acc = correct / total
    print(f"Validation accuracy: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_deepfake_model.pth")
        print(f"  -> New best model saved (val_acc={val_acc:.4f})")

print(f"\n🎉 Training complete. Best validation accuracy: {best_val_acc:.4f}")

# =====================================================================
# 10. Download the model
# =====================================================================
files.download("best_deepfake_model.pth")

📥 Downloading DeepDetect-2025...
DeepDetect path: /root/.cache/kagglehub/datasets/ayushmandatta1/deepdetect-2025/versions/1
📥 Downloading GRAVEX-200K...
GRAVEX path: /root/.cache/kagglehub/datasets/muhammadbilal6305/200k-real-vs-ai-visuals-by-mbilal/versions/4

🔍 DeepDetect structure:
/root/.cache/kagglehub/datasets/ayushmandatta1/deepdetect-2025/versions/1:
ddata

/root/.cache/kagglehub/datasets/ayushmandatta1/deepdetect-2025/versions/1/ddata:
test
train

/root/.cache/kagglehub/datasets/ayushmandatta1/deepdetect-2025/versions/1/ddata/test:
fake
real

/root/.cache/kagglehub/datasets/ayushmandatta1/deepdetect-2025/versions/1/ddata/test/fake:
fake_041594.jpg
fake_041595.jpg
fake_041596.jpg
fake_041597.jpg
fake_041598.jpg
fake_041599.jpg
fake_041600.jpg
fake_041601.jpg
fake_041602.jpg
fake_041603.jpg
fake_041604.jpg
fake_041605.jpg
fake_041606.jpg
fake_041607.jpg
fake_041608.jpg
fake_041609.jpg
fake_041610.jpg
fake_041611.jpg

🔍 GRAVEX structure:
/root/.cache/kagglehub/datasets/muhammadbi

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Epoch 1/10 [Train]: 100%|██████████| 3320/3320 [18:23<00:00,  3.01it/s, acc=0.916, loss=0.279]


Train accuracy: 0.9163


...

Train accuracy: 0.9854


Validation: 100%|██████████| 966/966 [03:24<00:00,  4.72it/s, acc=0.934]

Validation accuracy: 0.9340

🎉 Training complete. Best validation accuracy: 0.9535


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>